# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. We'll walk through metadata loading, examining available record sets and fields, data extraction, basic data processing, and visualization.

### Dataset Source
The dataset source is provided via this Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset's Croissant metadata and prepare for data exploration using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
# Access metadata as an object
meta = dataset.metadata

print(f"Dataset title: {meta.name}\n{meta.description}")

## 2. Data Overview

List available record sets, their `@id`s, and their fields. All references to entities are by their `@id` as required for cross-referencing and selection.

> **Tip:** If you see empty output, verify the schema URL and check that the dataset includes record sets.

In [ ]:
import pprint

# RecordSet objects in the dataset
record_sets = list(meta.record_sets)
print(f"Total record sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record set: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, Type: {field.data_type})")
    print("")

## 3. Data Extraction

Load data from one or more record sets into pandas DataFrames. In all code, refer to record sets and fields **by their `@id`**.

Below, we extract data for all available record sets.

In [ ]:
# Identify all record set @id(s)
record_set_ids = [rs.id for rs in meta.record_sets]
# Prepare a DataFrame for each record set
dataframes = {}

for rs_id in record_set_ids:
    # Load all records for this set
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} rows from record set '{rs_id}'. Columns: {list(df.columns)}\n")

# For demonstration, choose the first record set (if available)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Previewing columns for record set: {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)

For this step, select a numeric field by its `@id` for filtering and normalization. Adjust `numeric_field_id` and `group_field_id` to match field `@id`s extracted above for the chosen record set.

In [ ]:
# --- CONFIG ---
# Update these field @id values based on the prior overview cell's output:
record_set_id = main_record_set_id  # use the main record set
df = dataframes[record_set_id]

# Guess numeric and group field @id (change these as appropriate for your schema!):
# Demonstration: try field names resembling 'cr:MW' (Molecular Weight), 'cr:normalized_abundance', etc.
numeric_field_id = None
group_field_id = None

# Try to pick a numeric field automatically, fallback if not available
for col in df.columns:
    # Guessing numeric field
    if numeric_field_id is None and (
        'abundance' in col.lower() or 'mw' in col.lower() or 'count' in col.lower() or 'number' in col.lower()
    ):
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
    # Guess a group field
    if group_field_id is None and ('description' in col.lower() or 'group' in col.lower() or 'id' in col.lower()):
        group_field_id = col
if numeric_field_id is None:
    numeric_field_id = df.select_dtypes(include=['number']).columns[0] if not df.select_dtypes(include=['number']).empty else df.columns[0]
if group_field_id is None:
    # Pick any non-numeric field
    group_field_id = df.select_dtypes(exclude=['number']).columns[0] if not df.select_dtypes(exclude=['number']).empty else df.columns[0]

print(f"Using field '{numeric_field_id}' as numeric field for demonstration.")
print(f"Using field '{group_field_id}' as group field for demonstration.\n")

# Filter rows (e.g., abundance > threshold)
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
filtered_df = df[df[numeric_field_id] > threshold]

print(f"Filtered to rows with {numeric_field_id} > {threshold:.2f} (mean value): {len(filtered_df)} records.")
display(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (
    filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() != 0 else 1
)
print(f"\nNormalized {numeric_field_id} (z-score):")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field_id (if not too many groups)
if group_field_id in filtered_df.columns and filtered_df[group_field_id].nunique() < 50:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped average of {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization

Visualize the processed data distributions or group summaries to explore patterns or outliers.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the normalized numeric field
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=30, kde=True)
plt.title(f"Distribution of Normalized {numeric_field_id} (Record Set: {record_set_id})")
plt.xlabel(f"{numeric_field_id} (normalized)")
plt.ylabel("Frequency")
plt.show()

# If grouped data available, show a barplot of group averages
if 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(10, 4))
    grouped_df.sort_values(ascending=False).plot(kind='bar')
    plt.title(f"Mean {numeric_field_id} per {group_field_id}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(f"{group_field_id}")
    plt.show()

## 6. Conclusion

- We successfully loaded the FAIR^2 protein analysis dataset defined by a Croissant schema.
- Record sets, fields, and columns can be programmatically enumerated and referenced by their `@id`.
- We demonstrated basic numeric filtering, normalization, and grouping using selected field `@id`s.
- Data can be visualized directly once extracted into pandas, supporting further scientific analysis.

For more advanced processing, consult your dataset's field specification section and refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/api/mlcroissant/) for additional features!